# Marketing Campaign Data Cleaning

## Objective

This notebook cleans and standardizes the raw marketing campaign dataset based on the data-quality issues identified during profiling.

The cleaning process follows:

**Evidence → Decision → Transformation → Validation**

The raw dataset is preserved, while all transformations are performed on a working copy.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("marketing_campaign_data_messy.csv")

df_clean = df.copy()

In [2]:
df_clean.head()

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA


In [3]:
df_clean.tail()

,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA
2019,CMP-01554,Q4_Launch_CMP-01554,2023-06-26 00:00:00,2023-07-09,Email,96402,1090,1315.59,66.0,Y,NaN,EM


In [4]:
df_clean.shape

(2020, 12)

In [5]:
df_clean.describe()

,Impressions,Clicks,Conversions,Clicks
count,2020.000000,2020.000000,1820.000000,40.000000
mean,49839.896040,1500.744059,186.085714,54856.200000
std,28579.637473,1084.765654,160.129172,30552.773369
min,1055.000000,11.000000,0.000000,2508.000000
25%,25033.500000,650.750000,68.000000,30164.750000
50%,50097.500000,1245.000000,142.000000,57707.500000
75%,74784.250000,2185.250000,257.000000,81497.500000
max,99875.000000,4812.000000,943.000000,99483.000000


In [6]:
df_clean.info()
# Findings from df.info():
# - Total entries: 2020
# - Columns: 12
# - Data types: There are 8 object columns, 2 int64 columns, and 2 float64 columns.
# - Missing values:
#   - 'Channel' has 1919 non-null values, indicating 101 missing values.
#   - 'Conversions' has 1820 non-null values, indicating 200 missing values.
#   - The second 'Clicks' column (index 10) has only 40 non-null values, indicating 1980 missing values, confirming it is largely empty and likely a duplicate.
# - Data type considerations:
#   - 'Start_Date', 'End_Date', 'Spend', and 'Active' are object types but should likely be datetime or numeric/boolean types, requiring conversion.
#   - 'Campaign_ID', 'Campaign_Name', and 'Campaign_Tag' are object types, which is appropriate for categorical/identifier data.
# - Memory usage: 189.5+ KB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2020 entries, 0 to 2019
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0    Campaign_ID   2020 non-null   object 
 1   Campaign_Name  2020 non-null   object 
 2   Start_Date     2020 non-null   object 
 3   End_Date       2020 non-null   object 
 4   Channel        1919 non-null   object 
 5   Impressions    2020 non-null   int64  
 6   Clicks         2020 non-null   int64  
 7   Spend          2020 non-null   object 
 8   Conversions    1820 non-null   float64
 9   Active         2020 non-null   object 
 10  Clicks         40 non-null     float64
 11  Campaign_Tag   2020 non-null   object 
dtypes: float64(2), int64(2), object(8)
memory usage: 189.5+ KB


In [7]:
df_clean.isna().sum().sort_values(ascending=False)

,0
Clicks,1980
Conversions,200
Channel,101
Campaign_ID,0
End_Date,0
Start_Date,0
Campaign_Name,0
Impressions,0
Spend,0
Clicks,0


In [8]:
#Checking the name column inconsistency and duplicates
df_clean.columns

Index([' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel',
       'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks',
       'Campaign_Tag'],
      dtype='object')

In [9]:
# Renaming the both clicks columns for consistency.
df_clean = df_clean.rename(columns={
    "Clicks ": "clicks_1",
    "Clicks": "clicks_2"
})

In [10]:
#Checking inconsistencies in column headers
print(df_clean.columns.tolist())

#Fixing headers
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)
print(df_clean.columns)

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'clicks_1', 'Spend', 'Conversions', 'Active', 'clicks_2', 'Campaign_Tag']
Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks_1', 'spend', 'conversions', 'active', 'clicks_2',
       'campaign_tag'],
      dtype='object')


In [11]:
#Checking the number of unique campaigns run throughout 2023 - 2024.(Only 20 being repeated, need to inspect)
df_clean["campaign_id"].nunique()

2000

In [12]:
# Identifying the duplicate campaigns
print("Exact duplicate rows:", df.duplicated().sum())
print("Unique campaign IDs:", df[" Campaign_ID "].nunique())

Exact duplicate rows: 19
Unique campaign IDs: 2000


In [13]:
#Inspecting the duplicate 20 campaigns
duplicate_ids = df_clean[
    df_clean["campaign_id"].duplicated(keep=False)
].sort_values("campaign_id")

display(duplicate_ids)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks_1,spend,conversions,active,clicks_2,campaign_tag
184,CMP-00185,Q3_Launch_CMP-00185,2023-08-03 00:00:00,2023-08-27,Facebok,56016,1728,1890.33,128.0,No,NaN,FA
2014,CMP-00185,Q3_Launch_CMP-00185,2023-08-03 00:00:00,2023-08-27,Facebok,56016,1728,1890.33,128.0,No,NaN,FA
387,CMP-00388,Q1_Summer_CMP-00388,2023-06-20 00:00:00,2023-06-25,TikTok,57958,2328,$2135.23,280.0,Y,NaN,TI
2013,CMP-00388,Q1_Summer_CMP-00388,2023-06-20 00:00:00,2023-06-25,TikTok,57958,2328,$2135.23,280.0,Y,NaN,TI
399,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
438,CMP-00439,Q1_Launch_CMP-00439,2023-08-03 00:00:00,2023-08-27,Google Ads,88235,3182,5093.82,385.0,1,NaN,GO
2010,CMP-00439,Q1_Launch_CMP-00439,2023-08-03 00:00:00,2023-08-27,Google Ads,88235,3182,5093.82,385.0,1,NaN,GO
2011,CMP-01028,Q4_Summer_CMP-01028,2023-12-24 00:00:00,2024-01-05,TikTok,54441,2158,$2422.76,252.0,Yes,NaN,TI
1027,CMP-01028,Q4_Summer_CMP-01028,2023-12-24 00:00:00,2024-01-05,TikTok,54441,2158,$2422.76,252.0,Yes,NaN,TI


In [14]:
# 1. Checking the exact column names
print(df_clean.columns.tolist())

# 2. Find every row whose Campaign_ID appears more than once
duplicate_ids = df_clean[
    df_clean["campaign_id"].duplicated(keep=False)
].sort_values("campaign_id")

display(duplicate_ids)

['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks_1', 'spend', 'conversions', 'active', 'clicks_2', 'campaign_tag']


,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks_1,spend,conversions,active,clicks_2,campaign_tag
184,CMP-00185,Q3_Launch_CMP-00185,2023-08-03 00:00:00,2023-08-27,Facebok,56016,1728,1890.33,128.0,No,NaN,FA
2014,CMP-00185,Q3_Launch_CMP-00185,2023-08-03 00:00:00,2023-08-27,Facebok,56016,1728,1890.33,128.0,No,NaN,FA
387,CMP-00388,Q1_Summer_CMP-00388,2023-06-20 00:00:00,2023-06-25,TikTok,57958,2328,$2135.23,280.0,Y,NaN,TI
2013,CMP-00388,Q1_Summer_CMP-00388,2023-06-20 00:00:00,2023-06-25,TikTok,57958,2328,$2135.23,280.0,Y,NaN,TI
399,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
438,CMP-00439,Q1_Launch_CMP-00439,2023-08-03 00:00:00,2023-08-27,Google Ads,88235,3182,5093.82,385.0,1,NaN,GO
2010,CMP-00439,Q1_Launch_CMP-00439,2023-08-03 00:00:00,2023-08-27,Google Ads,88235,3182,5093.82,385.0,1,NaN,GO
2011,CMP-01028,Q4_Summer_CMP-01028,2023-12-24 00:00:00,2024-01-05,TikTok,54441,2158,$2422.76,252.0,Yes,NaN,TI
1027,CMP-01028,Q4_Summer_CMP-01028,2023-12-24 00:00:00,2024-01-05,TikTok,54441,2158,$2422.76,252.0,Yes,NaN,TI


In [15]:
# Columns used to compare campaign records
comparison_cols = [
    col for col in df_clean.columns
    if col.lower().strip() != "index"
]

# Count distinct versions of each repeated campaign
duplicate_audit = (
    df_clean[df_clean["campaign_id"].duplicated(keep=False)]
    .groupby("campaign_id")[comparison_cols]
    .apply(lambda group: len(group.drop_duplicates()))
    .reset_index(name="distinct_versions")
)

duplicate_audit["duplicate_type"] = duplicate_audit[
    "distinct_versions"
].map({
    1: "Exact duplicate",
}).fillna("Conflicting duplicate")

display(duplicate_audit)

,campaign_id,distinct_versions,duplicate_type
0,CMP-00185,1,Exact duplicate
1,CMP-00388,1,Exact duplicate
2,CMP-00400,1,Exact duplicate
3,CMP-00439,1,Exact duplicate
4,CMP-01028,1,Exact duplicate
5,CMP-01050,1,Exact duplicate
6,CMP-01064,1,Exact duplicate
7,CMP-01118,1,Exact duplicate
8,CMP-01200,1,Exact duplicate
9,CMP-01255,1,Exact duplicate


In [18]:
display(
    df_clean[df_clean["campaign_id"] == "CMP-01377"]
)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks_1,spend,conversions,active,clicks_2,campaign_tag
1376,CMP-01377,Q1_Winter_CMP-01377,2023-12-21 00:00:00,2023-12-29,E-mail,87143,2204,2649.14,391.0,Yes,NaN,INVALID
2008,CMP-01377,Q1_Winter_CMP-01377,2023-12-21 00:00:00,2023-12-29,E-mail,87143,2204,2649.14,391.0,Yes,NaN,E-


In [22]:
# Preserving the original tag before making corrections
df_clean["campaign_tag_raw"] = df_clean["campaign_tag"]

# Applying the documented mapping to this specific campaign
df_clean.loc[
    df_clean["campaign_id"] == "CMP-01377",
    "campaign_tag"
] = "EM"

In [23]:
# Verifying the rows before removing the duplicate campaigns.
print("Rows before:", len(df_clean))
print("Unique campaigns:", df_clean["campaign_id"].nunique())
print("Duplicate rows:", df_clean["campaign_id"].duplicated().sum())

Rows before: 2020
Unique campaigns: 2000
Duplicate rows: 20


In [24]:
# Removing the duplicated records
df_clean = df_clean.drop_duplicates(
    subset="campaign_id",
    keep="first"
 ).copy()

In [25]:
# Verifying the rows after removing the duplicate campaigns.
print("Rows after:", len(df_clean))
print("Unique campaigns:", df_clean["campaign_id"].nunique())

Rows after: 2000
Unique campaigns: 2000


Duplicate Campaign IDs: 20 campaign IDs appeared more than once. A record-level comparison showed that the duplicated rows contained identical values across all columns, confirming them as duplicate records rather than multiple observations of the same campaign. One copy of each duplicate record was retained.

`Campaign_ID - CMP-01377` - The duplicate campaign records have the same campaign information, and the only discrepancy is Campaign_Tag. Based on the observed Campaign_Tag–Channel relationship, E- is consistent with Email, while INVALID is inconsistent. Therefore, the two rows represent the same campaign record, and retaining one record is appropriate.

In [26]:
# Comparing the two Clicks columns

comparison = df_clean[["clicks_1", "clicks_2"]]

comparison.head(20)

,clicks_1,clicks_2
0,197,NaN
1,30,NaN
2,843,NaN
3,2019,NaN
4,169,NaN
5,2643,NaN
6,1135,NaN
7,1173,NaN
8,1179,NaN
9,1153,NaN


In [27]:
different_clicks = comparison[
    comparison["clicks_1"] != comparison["clicks_2"]
]

different_clicks.shape

(2000, 2)

In [28]:
different_clicks = comparison[
    ~comparison["clicks_1"].eq(comparison["clicks_2"])
    & ~(comparison["clicks_1"].isna() & comparison["clicks_2"].isna())
]

different_clicks.shape

(2000, 2)

In [29]:
clicks_1 = df_clean["clicks_1"]
clicks_2 = df_clean["clicks_2"]

print("Clicks 1 missing:", clicks_1.isna().sum())
print("Clicks 2 missing:", clicks_2.isna().sum())

Clicks 1 missing: 0
Clicks 2 missing: 1960


In [30]:
comparison[
    comparison["clicks_1"].isna() |
    comparison["clicks_2"].isna()
].head(20)

,clicks_1,clicks_2
0,197,NaN
1,30,NaN
2,843,NaN
3,2019,NaN
4,169,NaN
5,2643,NaN
6,1135,NaN
7,1173,NaN
8,1179,NaN
9,1153,NaN


In [31]:
both_present = comparison[
    comparison["clicks_1"].notna() &
    comparison["clicks_2"].notna()
]

(both_present["clicks_1"] == both_present["clicks_2"]).mean()

np.float64(0.0)

In [32]:
print("Clicks 1 missing:", clicks_1.isna().sum())
print("Clicks 2 missing:", clicks_2.isna().sum())

print("\nBoth present:")
print(both_present.shape)

print("\nAgreement when both are present:")
print((both_present["clicks_1"] == both_present["clicks_2"]).mean())

print("\nDifferent rows:")
print(different_clicks.shape)

Clicks 1 missing: 0
Clicks 2 missing: 1960

Both present:
(40, 2)

Agreement when both are present:
0.0

Different rows:
(2000, 2)


In [33]:
# The second Clicks column is largely missing and does not provide
# a reliable independent measure of clicks as per the profiling it also violates the business rule.

df_clean = df_clean.drop(columns="clicks_2")

In [36]:
#checking if the "clicks_2" column still exist
df_clean.columns

Index(['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel',
       'impressions', 'clicks_1', 'spend', 'conversions', 'active',
       'campaign_tag', 'campaign_tag_raw'],
      dtype='object')

Removed the duplicate `Clicks` columns with 40 nan values. Considering this data parsing issue with duplicates of the same column. The second Clicks column is largely missing and does not provide a reliable independent measure of clicks as per the profiling it also violates the business rule.

In [37]:
# 1. Checking the mixed values in spend
print(df_clean['spend'].head(20))

# 2. Clean unwanted characters
df_clean['spend'] = (
    df_clean['spend']
    .astype(str)
    .str.replace(r'[^\d.,-]', '', regex=True)
    .str.replace(',', '', regex=False)
)

# 3. Converting text to numeric
df_clean['spend'] = pd.to_numeric(df_clean['spend'], errors='coerce')

# 4. Checking the datatype
print(df_clean['spend'].dtype)

# 5. Investigating negative values separately
print(df_clean.loc[df_clean['spend'] < 0, ['campaign_id', 'spend']])

0      102.82
1       24.33
2     1323.39
3     2180.38
4      252.44
5     2697.03
6     1232.76
7      865.70
8     1046.18
9     1623.56
10     523.13
11    1267.69
12    5292.16
13    1464.72
14     139.18
15    1552.90
16    3147.37
17    3635.76
18    2933.59
19     353.99
Name: spend, dtype: float64
float64
     campaign_id    spend
144    CMP-00145 -1407.54
407    CMP-00408 -1214.52
576    CMP-00577  -821.80
604    CMP-00605  -942.80
676    CMP-00677 -1640.88
817    CMP-00818 -1261.81
886    CMP-00887 -2162.50
924    CMP-00925  -648.30
1125   CMP-01126  -122.80
1142   CMP-01143   -20.05
1317   CMP-01318 -1197.93
1409   CMP-01410  -386.43
1418   CMP-01419  -938.20
1457   CMP-01458  -551.19
1559   CMP-01560  -647.64
1701   CMP-01702 -1058.27
1710   CMP-01711   -65.19
1897   CMP-01898 -2503.31
1935   CMP-01936 -1697.64


In [38]:
#Checking the count of negative spend values
(df_clean['spend'] < 0).sum()

np.int64(19)

In [39]:
df_clean["spend_analysis"] = df_clean["spend"].where(
    df_clean["spend"] >= 0,
    np.nan
)

# Findings on Negative Spend Values:

*   The previous cell identified several campaigns with negative values in the 'spend' column (e.g., `CMP-00145` with -140754, `CMP-01898` with -250331).
*   Negative spend values are typically not expected for advertising campaigns and likely indicate data entry errors or specific accounting adjustments that need clarification.
*   These outliers could significantly skew aggregated spend analyses and impact the accuracy of campaign performance metrics if not handled appropriately.
*   Further investigation is needed to understand the cause of these negative values and determine the best approach for correction (e.g., setting to zero, taking absolute value, or removing the records if they are invalid).

In [40]:
# Inspecting the negative spend values

negative_spend = df_clean["spend"] < 0

print("Negative spend records:", negative_spend.sum())
print("Negative spend percentage:", negative_spend.mean()*100)

Negative spend records: 19
Negative spend percentage: 0.95


In [41]:
# checking the spend values against other columns to locate any relation

df_clean.loc[
    negative_spend,
    ["campaign_id", "campaign_name", "start_date", "end_date", "spend", "impressions", "clicks_1", "conversions"]
].head(20)

,campaign_id,campaign_name,start_date,end_date,spend,impressions,clicks_1,conversions
144,CMP-00145,Q2_Summer_CMP-00145,2023-12-07 00:00:00,2023-12-14,-1407.54,51636,1859,270.0
407,CMP-00408,Q1_BlackFriday_CMP-00408,2023-05-21 00:00:00,2023-05-28,-1214.52,78575,2058,119.0
576,CMP-00577,Q4_Launch_CMP-00577,2023-06-30 00:00:00,2023-07-13,-821.80,90084,940,178.0
604,CMP-00605,Q2_BlackFriday_CMP-00605,2023-02-03 00:00:00,2023-02-25,-942.80,42555,515,57.0
676,CMP-00677,Q3_BlackFriday_CMP-00677,2023-10-15 00:00:00,2023-10-26,-1640.88,41262,1025,64.0
817,CMP-00818,Q1_Launch_CMP-00818,2023-09-05,2023-05-16,-1261.81,35816,1291,82.0
886,CMP-00887,Q4_Summer_CMP-00887,2023-07-29 00:00:00,2023-08-13,-2162.50,70836,2292,150.0
924,CMP-00925,Q4_Launch_CMP-00925,2023-02-28 00:00:00,2023-03-15,-648.30,23574,1054,104.0
1125,CMP-01126,Q1_Launch_CMP-01126,2023-11-11 00:00:00,2023-12-11,-122.80,12302,204,25.0
1142,CMP-01143,Q1_BlackFriday_CMP-01143,2023-11-10 00:00:00,2023-12-01,-20.05,1922,29,2.0


In [42]:
df_clean.loc[
    negative_spend,
    ["conversions"]
].head(20)

,conversions
144,270.0
407,119.0
576,178.0
604,57.0
676,64.0
817,82.0
886,150.0
924,104.0
1125,25.0
1142,2.0


In [43]:
df_clean.loc[negative_spend, "spend"].describe()

,spend
count,19.000000
mean,-1015.200000
std,678.882126
min,-2503.310000
25%,-1334.675000
50%,-942.800000
75%,-599.415000
max,-20.050000


In [44]:
df_clean.loc[negative_spend,
       ["campaign_id", "campaign_name", "channel", "spend"]
].sort_values("spend")

,campaign_id,campaign_name,channel,spend
1897,CMP-01898,Q4_Launch_CMP-01898,NaN,-2503.31
886,CMP-00887,Q4_Summer_CMP-00887,Google Ads,-2162.50
1935,CMP-01936,Q1_Launch_CMP-01936,Facebook,-1697.64
676,CMP-00677,Q3_BlackFriday_CMP-00677,TikTok,-1640.88
144,CMP-00145,Q2_Summer_CMP-00145,Facebook,-1407.54
817,CMP-00818,Q1_Launch_CMP-00818,Email,-1261.81
407,CMP-00408,Q1_BlackFriday_CMP-00408,Facebok,-1214.52
1317,CMP-01318,Q2_Launch_CMP-01318,TikTok,-1197.93
1701,CMP-01702,Q4_Summer_CMP-01702,Facebook,-1058.27
604,CMP-00605,Q2_BlackFriday_CMP-00605,Email,-942.80


In [45]:
df_clean.loc[negative_spend, "channel"].value_counts(dropna=False)

,count
channel,
Facebook,6
TikTok,4
Google Ads,3
NaN,2
Email,2
Facebok,1
Instagram,1


In [46]:
df_clean["spend_valid_flag"] = (
    df_clean["spend"] >= 0
)

In [47]:
df_clean["negative_spend_flag"] = df_clean["spend"] < 0

In [152]:
#Creating a new spend column for analysis without the negative values
df_clean["spend_clean"] = df_clean["spend"].where(
    df_clean["spend"] >= 0,
    pd.NA
)

In [153]:
df_clean.loc[
    negative_spend,
    ["campaign_id", "spend", "impressions", "clicks_1", "conversions"]
]

,campaign_id,spend,impressions,clicks_1,conversions
144,CMP-00145,-1407.54,51636,1859,270.0
407,CMP-00408,-1214.52,78575,2058,119.0
576,CMP-00577,-821.80,90084,940,178.0
604,CMP-00605,-942.80,42555,515,57.0
676,CMP-00677,-1640.88,41262,1025,64.0
817,CMP-00818,-1261.81,35816,1291,82.0
886,CMP-00887,-2162.50,70836,2292,150.0
924,CMP-00925,-648.30,23574,1054,104.0
1125,CMP-01126,-122.80,12302,204,25.0
1142,CMP-01143,-20.05,1922,29,2.0


Negative Spend values were identified and treated as data-quality anomalies rather than automatically imputed. Since the Kaggle dataset does not provide source-level documentation confirming whether negative values represent legitimate refunds/adjustments, the original values were retained and flagged for transparency. Records were excluded from expenditure-based calculations where negative spend would distort the business metric.


A negative spend could potentially represent something like:

1. refund/reversal
2. billing adjustment
3. credit
4. correction to previously recorded spend

Since, this is a Kaggle dataset defines Spend as campaign expenditure, then a negative value may instead indicate a data-generation or data-entry problem. There seems a clean relation of the spend column with clicks and channels. In all four possible listed above needs to be verified at source and cannot be directly assumed.

In [48]:
# Fixing Active column with inconsistencies
print(df_clean["channel"].unique())

cleaned_channel = {
    "Tik_Tok": "Tiktok",
    "TikTok": "Tiktok",
    "Facebok": "Facebook",
    "E-mail": "Email",
    "Gogle": "Google Ads",
    "Insta_gram": "Instagram",
    "N/A": np.nan #replacing nan values to null
}

df_clean["channel"] = df_clean["channel"].replace(cleaned_channel)
print(df_clean["channel"])

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
0           Tiktok
1         Facebook
2            Email
3           Tiktok
4         Facebook
           ...    
1995    Google Ads
1996      Facebook
1997     Instagram
1998     Instagram
1999        Tiktok
Name: channel, Length: 2000, dtype: object


In [49]:
#checking the dtype for channel
df_clean["channel"].dtype

dtype('O')

In [50]:
#Handling mixed values

active_mapping = {
    "Y": True,
    "YES": True,
    "TRUE": True,
    "1": True,
    "NO": False,
    "FALSE": False,
    "0": False
}

df_clean["active"] = (
    df_clean["active"]
    .astype("string")
    .str.strip()
    .str.upper()
    .map(active_mapping)
)

print(df_clean["active"].value_counts(dropna=False))
print(df_clean["active"].dtype)

active
True     1182
False     818
Name: count, dtype: int64
bool


In [51]:
#Extracting the seasons as campaign type from campaign name
df_clean["campaign_type"] = (
    df_clean["campaign_name"]
    .str.split("_")
    .str[1]
)

In [52]:
#checking unique campaign type
df_clean["campaign_type"].unique()

array(['Summer', 'Launch', 'Winter', 'BlackFriday'], dtype=object)

In [53]:
df_clean["campaign_tag"].unique()

array(['TI', 'FA', 'EM', 'IN', 'GO', 'E-', 'INVALID', 'XX'], dtype=object)

In [54]:
#checking if there any relation between campaign_type and campaign_tag to correct the inconsistencies in the campaign_tag column.
pd.crosstab(
    df_clean["campaign_type"],
    df_clean["campaign_tag"]
)

campaign_tag,E-,EM,FA,GO,IN,INVALID,TI,XX
campaign_type,,,,,,,,
BlackFriday,10,78,104,76,92,14,107,22
Launch,3,79,92,89,88,31,105,19
Summer,3,81,88,85,78,32,82,30
Winter,3,94,103,81,84,23,103,21


In [55]:
#checking the relation between campaign_tag and channel

pd.crosstab(
    df_clean["campaign_tag"],
    df_clean["channel"]
)

channel,Email,Facebook,Google Ads,Instagram,Tiktok
campaign_tag,,,,,
E-,19,0,0,0,0
EM,332,0,0,0,0
FA,0,387,0,0,0
GO,0,0,331,0,0
IN,0,0,0,342,0
INVALID,24,14,23,16,15
TI,0,0,0,0,397


In [56]:
channel_to_tag = {
    "Email": "EM",
    "Facebook": "FA",
    "Google Ads": "GO",
    "Instagram": "IN",
    "Tiktok": "TI"
}

df_clean.loc[
    df_clean["campaign_tag"] == "INVALID",
    "campaign_tag"
] = (
    df_clean.loc[
        df_clean["campaign_tag"] == "INVALID",
        "channel"
    ].map(channel_to_tag)
)

In [57]:
df_clean["campaign_tag"] = df_clean["campaign_tag"].replace({
    "E-": "EM"
})

In [58]:
df_clean["campaign_tag"].unique()

array(['TI', 'FA', 'EM', 'IN', 'GO', 'XX', nan], dtype=object)

In [59]:
df_clean[df_clean["channel"] == "Email"][
    ["campaign_id", "campaign_name", "channel", "campaign_tag"]
].query("campaign_tag == 'E-'").head(20)

,campaign_id,campaign_name,channel,campaign_tag


In [60]:
#checking the values have been correctly assigned to campaign_tag as per channel column
pd.crosstab(
    df_clean["campaign_tag"],
    df_clean["channel"]
)

channel,Email,Facebook,Google Ads,Instagram,Tiktok
campaign_tag,,,,,
EM,375,0,0,0,0
FA,0,401,0,0,0
GO,0,0,354,0,0
IN,0,0,0,358,0
TI,0,0,0,0,412


In [61]:
df_clean[df_clean["campaign_tag"] == "INVALID"].head(20)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks_1,spend,conversions,active,campaign_tag,campaign_tag_raw,spend_analysis,spend_valid_flag,negative_spend_flag,campaign_type


In [62]:
df_clean[df_clean["campaign_tag"] == "INVALID"][
    ["channel", "campaign_tag"]
].value_counts()

,,count
channel,campaign_tag,


In [63]:
df_clean[df_clean["campaign_tag"] == "INVALID"][
    ["campaign_id", "channel", "campaign_tag"]
].head(20)

,campaign_id,channel,campaign_tag


In [64]:
df_clean["campaign_tag_valid_flag"] = (
    df_clean["campaign_tag"].notna() &
    (df_clean["campaign_tag"] != "INVALID")
)

In [65]:
xx_records = df_clean[
    df_clean["campaign_tag"] == "XX"
]

xx_records

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks_1,spend,conversions,active,campaign_tag,campaign_tag_raw,spend_analysis,spend_valid_flag,negative_spend_flag,campaign_type,campaign_tag_valid_flag
53,CMP-00054,Q4_Winter_CMP-00054,2023-09-30 00:00:00,2023-10-11,NaN,49555,1114,919.99,95.0,False,XX,XX,919.99,True,False,Winter,True
88,CMP-00089,Q4_Winter_CMP-00089,2023-09-29 00:00:00,2023-10-10,NaN,87807,3125,1938.54,506.0,True,XX,XX,1938.54,True,False,Winter,True
93,CMP-00094,Q1_Winter_CMP-00094,2023-08-03 00:00:00,2023-08-29,NaN,9571,319,497.40,52.0,False,XX,XX,497.40,True,False,Winter,True
97,CMP-00098,Q4_Summer_CMP-00098,07/02/2023,2023-02-26,NaN,49984,1871,950.50,273.0,False,XX,XX,950.50,True,False,Summer,True
102,CMP-00103,Q3_Summer_CMP-00103,2023-10-06 00:00:00,2023-11-04,NaN,3695,74,44.70,7.0,False,XX,XX,44.70,True,False,Summer,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1930,CMP-01931,Q4_Launch_CMP-01931,2023-10-19,2023-10-31,NaN,23431,951,1512.39,179.0,False,XX,XX,1512.39,True,False,Launch,True
1951,CMP-01952,Q1_Summer_CMP-01952,2023-03-20 00:00:00,2023-04-08,NaN,26003,812,414.70,120.0,False,XX,XX,414.70,True,False,Summer,True
1962,CMP-01963,Q2_BlackFriday_CMP-01963,2023-04-13 00:00:00,2023-04-14,NaN,91827,1072,1558.54,208.0,True,XX,XX,1558.54,True,False,BlackFriday,True
1973,CMP-01974,Q3_Winter_CMP-01974,2023-07-20 00:00:00,2023-08-08,NaN,67435,3216,2147.97,302.0,True,XX,XX,2147.97,True,False,Winter,True


In [66]:
xx_records = df_clean[
    df_clean["campaign_tag"] == "XX"
]

xx_records[
    [
        "campaign_id",
        "campaign_name",
        "campaign_type",
        "channel",
        "campaign_tag"
    ]
]

,campaign_id,campaign_name,campaign_type,channel,campaign_tag
53,CMP-00054,Q4_Winter_CMP-00054,Winter,NaN,XX
88,CMP-00089,Q4_Winter_CMP-00089,Winter,NaN,XX
93,CMP-00094,Q1_Winter_CMP-00094,Winter,NaN,XX
97,CMP-00098,Q4_Summer_CMP-00098,Summer,NaN,XX
102,CMP-00103,Q3_Summer_CMP-00103,Summer,NaN,XX
...,...,...,...,...,...
1930,CMP-01931,Q4_Launch_CMP-01931,Launch,NaN,XX
1951,CMP-01952,Q1_Summer_CMP-01952,Summer,NaN,XX
1962,CMP-01963,Q2_BlackFriday_CMP-01963,BlackFriday,NaN,XX
1973,CMP-01974,Q3_Winter_CMP-01974,Winter,NaN,XX


In [67]:
xx_records["campaign_type"].value_counts(dropna=False)

,count
campaign_type,
Summer,30
BlackFriday,22
Winter,21
Launch,19


In [68]:
pd.crosstab(
    df_clean["campaign_type"],
    df_clean["campaign_tag"],
    dropna=False
)

campaign_tag,EM,FA,GO,IN,TI,XX,NaN
campaign_type,,,,,,,
BlackFriday,89,106,79,95,111,22,1
Launch,94,94,96,92,107,19,4
Summer,93,95,87,84,88,30,2
Winter,99,106,92,87,106,21,1


In [69]:
missing_tag = df_clean[
    df_clean["campaign_tag"].isna()
]

missing_tag[
    [
        "campaign_id",
        "campaign_name",
        "campaign_type",
        "channel",
        "campaign_tag"
    ]
]

,campaign_id,campaign_name,campaign_type,channel,campaign_tag
162,CMP-00163,Q4_BlackFriday_CMP-00163,BlackFriday,NaN,NaN
208,CMP-00209,Q2_Launch_CMP-00209,Launch,NaN,NaN
224,CMP-00225,Q2_Launch_CMP-00225,Launch,NaN,NaN
403,CMP-00404,Q4_Launch_CMP-00404,Launch,NaN,NaN
590,CMP-00591,Q3_Summer_CMP-00591,Summer,NaN,NaN
713,CMP-00714,Q1_Winter_CMP-00714,Winter,NaN,NaN
1073,CMP-01074,Q4_Summer_CMP-01074,Summer,NaN,NaN
1744,CMP-01745,Q4_Launch_CMP-01745,Launch,NaN,NaN


In [70]:
valid_tags = df_clean[
    ~df_clean["campaign_tag"].isin(["INVALID", "XX"]) &
    df_clean["campaign_tag"].notna()
]

pd.crosstab(
    valid_tags["campaign_type"],
    valid_tags["campaign_tag"]
)

campaign_tag,EM,FA,GO,IN,TI
campaign_type,,,,,
BlackFriday,89,106,79,95,111
Launch,94,94,96,92,107
Summer,93,95,87,84,88
Winter,99,106,92,87,106


In [71]:
missing_tag = df_clean[
    df_clean["campaign_tag"].isna()
]

In [129]:
missing_tag[
    [
        "campaign_id",
        "campaign_name",
        "campaign_type",
        "channel",
        "campaign_tag"
    ]
]

,campaign_id,campaign_name,campaign_type,channel,campaign_tag
162,CMP-00163,Q4_BlackFriday_CMP-00163,BlackFriday,NaN,NaN
208,CMP-00209,Q2_Launch_CMP-00209,Launch,NaN,NaN
224,CMP-00225,Q2_Launch_CMP-00225,Launch,NaN,NaN
403,CMP-00404,Q4_Launch_CMP-00404,Launch,NaN,NaN
590,CMP-00591,Q3_Summer_CMP-00591,Summer,NaN,NaN
713,CMP-00714,Q1_Winter_CMP-00714,Winter,NaN,NaN
1073,CMP-01074,Q4_Summer_CMP-01074,Summer,NaN,NaN
1744,CMP-01745,Q4_Launch_CMP-01745,Launch,NaN,NaN


In [72]:
missing_tag["campaign_type"].value_counts(dropna=False)

,count
campaign_type,
Launch,4
Summer,2
BlackFriday,1
Winter,1


In [73]:
xx_records["channel"].value_counts(dropna=False)

,count
channel,
NaN,92


In [74]:
pd.crosstab(
    df_clean["campaign_tag"],
    df_clean["channel"],
    dropna=False
)

channel,Email,Facebook,Google Ads,Instagram,Tiktok,NaN
campaign_tag,,,,,,
EM,375,0,0,0,0,0
FA,0,401,0,0,0,0
GO,0,0,354,0,0,0
IN,0,0,0,358,0,0
TI,0,0,0,0,412,0
XX,0,0,0,0,0,92
NaN,0,0,0,0,0,8


In [75]:
pd.crosstab(
    df_clean["campaign_tag"],
    df_clean["channel"],
    dropna=False
).loc["XX"]

,XX
channel,
Email,0
Facebook,0
Google Ads,0
Instagram,0
Tiktok,0
NaN,92


In [76]:
xx_records = df_clean[df_clean["campaign_tag"] == "XX"]

print("XX records:", len(xx_records))
print("\nChannels:")
print(xx_records["channel"].value_counts(dropna=False))

print("\nMissing channel:", xx_records["channel"].isna().sum())

XX records: 92

Channels:
channel
NaN    92
Name: count, dtype: int64

Missing channel: 92


In [77]:
df_clean.groupby(["channel", "campaign_tag"]).size().reset_index(name="count")

,channel,campaign_tag,count
0,Email,EM,375
1,Facebook,FA,401
2,Google Ads,GO,354
3,Instagram,IN,358
4,Tiktok,TI,412


In [78]:
df_clean.groupby("channel")["campaign_tag"].value_counts()

,,count
channel,campaign_tag,
Email,EM,375
Facebook,FA,401
Google Ads,GO,354
Instagram,IN,358
Tiktok,TI,412


In [79]:
df_clean["campaign_tag_valid_flag"] = (
    df_clean["campaign_tag"].notna() &
    (df_clean["campaign_tag"] != "INVALID")
)

In [80]:
df_clean["campaign_tag"] = df_clean["campaign_tag"].replace({
    "E-": "EM"
})

In [81]:
df_clean[df_clean["channel"] == "Email"]["campaign_tag"].value_counts()

,count
campaign_tag,
EM,375


In [82]:
df_clean["campaign_tag"].value_counts(dropna=False)

,count
campaign_tag,
TI,412
FA,401
EM,375
IN,358
GO,354
XX,92
NaN,8


In [83]:
#Replacing "Invalid" and "XX" values in campaign_tag column.
df_clean["campaign_tag"] = (
    df_clean["campaign_tag"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_clean["campaign_tag"] = df_clean["campaign_tag"].replace(
    {"INVALID": "Unknown", "XX": "Unknown"}
)

df_clean["campaign_tag"] = df_clean["campaign_tag"].fillna("Unknown")

In [84]:
df_clean["campaign_tag"].value_counts(dropna=False)

,count
campaign_tag,
TI,412
FA,401
EM,375
IN,358
GO,354
Unknown,100


In [85]:
#Checking if there is any missing pattern in channel column for nan values.

# 1. Creating a flag for missing channel values
df_clean["channel_missing_flag"] = df_clean["channel"].isna()

# 2. Establishing the baseline
total_records = len(df_clean)
missing_channel_count = df_clean["channel_missing_flag"].sum()
missing_channel_pct = (
    df_clean["channel_missing_flag"].mean() * 100
)

print("Total records:", total_records)
print("Missing channel records:", missing_channel_count)
print(f"Missing channel percentage: {missing_channel_pct:.2f}%")
print("Non-missing channel records:",
      (~df_clean["channel_missing_flag"]).sum())

Total records: 2000
Missing channel records: 100
Missing channel percentage: 5.00%
Non-missing channel records: 1900


In [86]:
#Checking if there is any missing pattern in channel column for nan values.
categorical_columns = [
    "campaign_tag",
    "campaign_type",
    "active"
]

for col in categorical_columns:
    if col not in df_clean.columns:
        continue

    summary = (
        df_clean.groupby(col, dropna=False)["channel_missing_flag"]
        .agg(
            total_records="size",
            missing_channel_records="sum",
            missing_channel_rate="mean"
        )
        .reset_index()
    )

    summary["missing_channel_rate"] = (
        summary["missing_channel_rate"] * 100
    ).round(2)

    summary = summary.sort_values(
        "missing_channel_rate",
        ascending=False
    )

    print(f"\nMissing channel pattern by {col}")
    display(summary)


Missing channel pattern by campaign_tag


,campaign_tag,total_records,missing_channel_records,missing_channel_rate
5,Unknown,100,100,100.0
0,EM,375,0,0.0
1,FA,401,0,0.0
2,GO,354,0,0.0
3,IN,358,0,0.0
4,TI,412,0,0.0



Missing channel pattern by campaign_type


,campaign_type,total_records,missing_channel_records,missing_channel_rate
2,Summer,479,32,6.68
0,BlackFriday,503,23,4.57
1,Launch,506,23,4.55
3,Winter,512,22,4.30



Missing channel pattern by active


,active,total_records,missing_channel_records,missing_channel_rate
1,True,1182,65,5.50
0,False,818,35,4.28


In [87]:
# checking for any missing pattern in channel column against numeric columns.
numeric_columns = [
    "impressions",
    "clicks_1",
    "conversions",
    "spend"
]

available_numeric_columns = [
    col for col in numeric_columns
    if col in df_clean.columns
]

numeric_comparison = (
    df_clean.groupby("channel_missing_flag")[available_numeric_columns]
    .agg(["count", "median", "mean", "std"])
    .round(2)
)

numeric_comparison.index = [
    "Channel Missing" if flag else "Channel Present"
    for flag in numeric_comparison.index
]

display(numeric_comparison)

impressions                              clicks_1          \
                      count   median      mean       std    count  median   
Channel Present        1900  50145.5  50030.47  28569.26     1900  1246.5   
Channel Missing         100  46237.5  46421.69  29180.04      100  1104.5   

                                  conversions                        spend  \
                    mean      std       count median    mean     std count   
Channel Present  1500.95  1080.07        1708  141.0  185.78  159.83  1900   
Channel Missing  1506.66  1212.85          92  147.0  192.87  170.69   100   

                                             
                  median     mean       std  
Channel Present  1432.70  3181.91  25577.60  
Channel Missing  1244.39  1698.91   1576.42

In [88]:

columns_to_inspect = [
    "campaign_id",
    "campaign_name",
    "campaign_type",
    "start_date",
    "end_date",
    "channel",
    "campaign_tag",
    "active",
    "impressions",
    "clicks_1",
    "conversions",
    "spend"
]

columns_to_inspect = [
    col for col in columns_to_inspect
    if col in df_clean.columns
]

missing_channel_records = df_clean.loc[
    df_clean["channel_missing_flag"],
    columns_to_inspect
].copy()

print("Records with missing channel:", len(missing_channel_records))

display(
    missing_channel_records.sort_values(
        by=["campaign_tag", "campaign_type"],
        na_position="last"
    )
)

Records with missing channel: 100


,campaign_id,campaign_name,campaign_type,start_date,end_date,channel,campaign_tag,active,impressions,clicks_1,conversions,spend
162,CMP-00163,Q4_BlackFriday_CMP-00163,BlackFriday,2023-07-24 00:00:00,2023-08-13,NaN,Unknown,True,43557,1386,151.0,2596.68
170,CMP-00171,Q2_BlackFriday_CMP-00171,BlackFriday,2023-09-16 00:00:00,2023-09-27,NaN,Unknown,True,43941,1392,183.0,1479.92
437,CMP-00438,Q1_BlackFriday_CMP-00438,BlackFriday,2023-02-16 00:00:00,2023-03-07,NaN,Unknown,True,88142,1739,249.0,1227.10
490,CMP-00491,Q1_BlackFriday_CMP-00491,BlackFriday,2023-05-25 00:00:00,2023-06-02,NaN,Unknown,True,94468,4632,400.0,2847.10
562,CMP-00563,Q4_BlackFriday_CMP-00563,BlackFriday,2023-10-06,2023-07-03,NaN,Unknown,True,77797,1381,192.0,1342.66
...,...,...,...,...,...,...,...,...,...,...,...,...
1617,CMP-01618,Q3_Winter_CMP-01618,Winter,2023-07-25 00:00:00,2023-08-03,NaN,Unknown,True,81688,3367,613.0,4720.79
1633,CMP-01634,Q1_Winter_CMP-01634,Winter,2023-04-12 00:00:00,2023-04-07,NaN,Unknown,False,81435,1509,294.0,2104.53
1779,CMP-01780,Q3_Winter_CMP-01780,Winter,2023-04-27 00:00:00,2023-05-18,NaN,Unknown,True,92357,4604,594.0,5093.69
1865,CMP-01866,Q3_Winter_CMP-01866,Winter,2023-05-06 00:00:00,2023-05-17,NaN,Unknown,True,6776,239,NaN,477.11


In [89]:

channel_missing_by_tag = (
    df_clean.groupby("campaign_tag", dropna=False)
    .agg(
        total_records=("channel_missing_flag", "size"),
        missing_channel_records=("channel_missing_flag", "sum")
    )
)

channel_missing_by_tag["missing_channel_rate_pct"] = (
    channel_missing_by_tag["missing_channel_records"]
    / channel_missing_by_tag["total_records"]
    * 100
).round(2)

display(
    channel_missing_by_tag.sort_values(
        "missing_channel_rate_pct",
        ascending=False
    )
)

,total_records,missing_channel_records,missing_channel_rate_pct
campaign_tag,,,
Unknown,100,100,100.0
EM,375,0,0.0
FA,401,0,0.0
GO,354,0,0.0
IN,358,0,0.0
TI,412,0,0.0


In [90]:

# Cross-tabulate channel missingness by campaign tag and campaign type
if {"campaign_tag", "campaign_type"}.issubset(df_clean.columns):

    pattern_summary = (
        df_clean.groupby(
            ["campaign_tag", "campaign_type"],
            dropna=False
        )["channel_missing_flag"]
        .agg(
            total_records="size",
            missing_channel_records="sum",
            missing_channel_rate="mean"
        )
        .reset_index()
    )

    pattern_summary["missing_channel_rate"] = (
        pattern_summary["missing_channel_rate"] * 100
    ).round(2)

    display(
        pattern_summary.sort_values(
            ["missing_channel_rate", "total_records"],
            ascending=[False, False]
        )
    )

,campaign_tag,campaign_type,total_records,missing_channel_records,missing_channel_rate
22,Unknown,Summer,32,32,100.0
20,Unknown,BlackFriday,23,23,100.0
21,Unknown,Launch,23,23,100.0
23,Unknown,Winter,22,22,100.0
16,TI,BlackFriday,111,0,0.0
17,TI,Launch,107,0,0.0
4,FA,BlackFriday,106,0,0.0
7,FA,Winter,106,0,0.0
19,TI,Winter,106,0,0.0
3,EM,Winter,99,0,0.0


In [91]:

# Count records for each campaign type and channel combination
campaign_channel_counts = (
    df_clean.groupby(
        ["campaign_type", "channel"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["campaign_type", "record_count"],
        ascending=[True, False]
    )
)

display(campaign_channel_counts)

,campaign_type,channel,record_count
4,BlackFriday,Tiktok,111
1,BlackFriday,Facebook,106
3,BlackFriday,Instagram,95
0,BlackFriday,Email,89
2,BlackFriday,Google Ads,79
5,BlackFriday,NaN,23
10,Launch,Tiktok,107
8,Launch,Google Ads,96
6,Launch,Email,94
7,Launch,Facebook,94


In [92]:

campaign_channel_matrix = pd.crosstab(
    df_clean["campaign_type"],
    df_clean["channel"],
    dropna=False
)

display(campaign_channel_matrix)

channel,Email,Facebook,Google Ads,Instagram,Tiktok,NaN
campaign_type,,,,,,
BlackFriday,89,106,79,95,111,23
Launch,94,94,96,92,107,23
Summer,93,95,87,84,88,32
Winter,99,106,92,87,106,22


In [93]:

missing_channel_by_type = (
    df_clean.groupby("campaign_type", dropna=False)
    .agg(
        total_records=("channel", "size"),
        missing_channel_records=("channel", lambda x: x.isna().sum())
    )
)

missing_channel_by_type["missing_channel_pct"] = (
    missing_channel_by_type["missing_channel_records"]
    / missing_channel_by_type["total_records"]
    * 100
).round(2)

display(
    missing_channel_by_type.sort_values(
        "missing_channel_pct",
        ascending=False
    )
)

,total_records,missing_channel_records,missing_channel_pct
campaign_type,,,
Summer,479,32,6.68
BlackFriday,503,23,4.57
Launch,506,23,4.55
Winter,512,22,4.30


In [94]:
# Inspect Summer campaigns with missing channel values
summer_missing_channel = df_clean[
    (df_clean["campaign_type"] == "Summer")
    & (df_clean["channel"].isna())
]

columns_to_check = [
    "campaign_id",
    "campaign_name",
    "campaign_type",
    "channel",
    "campaign_tag",
    "active",
    "start_date",
    "end_date",
    "impressions",
    "clicks_1",
    "conversions",
    "spend"
]

columns_to_check = [
    col for col in columns_to_check
    if col in df_clean.columns
]

display(summer_missing_channel[columns_to_check])

,campaign_id,campaign_name,campaign_type,channel,campaign_tag,active,start_date,end_date,impressions,clicks_1,conversions,spend
97,CMP-00098,Q4_Summer_CMP-00098,Summer,NaN,Unknown,False,07/02/2023,2023-02-26,49984,1871,273.0,950.50
102,CMP-00103,Q3_Summer_CMP-00103,Summer,NaN,Unknown,False,2023-10-06 00:00:00,2023-11-04,3695,74,7.0,44.70
149,CMP-00150,Q2_Summer_CMP-00150,Summer,NaN,Unknown,True,2023-07-14 00:00:00,2023-07-19,81356,3670,660.0,5352.87
231,CMP-00232,Q2_Summer_CMP-00232,Summer,NaN,Unknown,True,2023-12-06 00:00:00,2023-12-15,26342,1074,81.0,2122.60
236,CMP-00237,Q2_Summer_CMP-00237,Summer,NaN,Unknown,True,2023-12-20 00:00:00,2023-12-30,90045,3966,587.0,3117.63
284,CMP-00285,Q2_Summer_CMP-00285,Summer,NaN,Unknown,True,2023-05-10 00:00:00,2023-05-13,73694,1417,177.0,1245.82
322,CMP-00323,Q2_Summer_CMP-00323,Summer,NaN,Unknown,True,2023-07-14 00:00:00,2023-08-13,47214,694,106.0,738.21
326,CMP-00327,Q2_Summer_CMP-00327,Summer,NaN,Unknown,False,2023-08-11 00:00:00,2023-08-19,71091,1695,261.0,2787.16
383,CMP-00384,Q2_Summer_CMP-00384,Summer,NaN,Unknown,True,2023-04-08 00:00:00,2023-05-05,5014,163,21.0,214.27
460,CMP-00461,Q2_Summer_CMP-00461,Summer,NaN,Unknown,True,2023-08-24 00:00:00,2023-08-31,33217,930,158.0,593.57


In [95]:

summer_df = df_clean[
    df_clean["campaign_type"] == "Summer"
].copy()

for col in ["campaign_tag", "active"]:
    if col not in summer_df.columns:
        continue

    summary = (
        summer_df.groupby(col, dropna=False)["channel"]
        .agg(
            total_records="size",
            missing_channels=lambda x: x.isna().sum()
        )
    )

    summary["missing_channel_pct"] = (
        summary["missing_channels"]
        / summary["total_records"]
        * 100
    ).round(2)

    print(f"\nSummer campaign missingness by {col}:")
    display(summary.sort_values("missing_channel_pct", ascending=False))


Summer campaign missingness by campaign_tag:


,total_records,missing_channels,missing_channel_pct
campaign_tag,,,
Unknown,32,32,100.0
EM,93,0,0.0
FA,95,0,0.0
GO,87,0,0.0
IN,84,0,0.0
TI,88,0,0.0



Summer campaign missingness by active:


,total_records,missing_channels,missing_channel_pct
active,,,
True,303,22,7.26
False,176,10,5.68


In [96]:
#Replacing nan channel values to "Unknown"
df_clean["channel"] = (
    df_clean["channel"]
    .astype("string")
    .str.strip()
)

df_clean["channel"] = df_clean["channel"].fillna("Unknown")

In [97]:
df_clean["channel"].value_counts(dropna=False)

,count
channel,
Tiktok,412
Facebook,401
Email,375
Instagram,358
Google Ads,354
Unknown,100


In [98]:
conversion_missing = df_clean[
    df_clean["conversions"].isna()
]

In [99]:
conversion_missing[
    [
        "campaign_id",
        "campaign_name",
        "campaign_type",
        "channel",
        "impressions",
        "clicks_1",
        "spend",
        "active",
        "campaign_tag",
        "conversions"
    ]
]

,campaign_id,campaign_name,campaign_type,channel,impressions,clicks_1,spend,active,campaign_tag,conversions
5,CMP-00006,Q4_BlackFriday_CMP-00006,BlackFriday,Instagram,83386,2643,2697.03,True,IN,NaN
9,CMP-00010,Q2_Winter_CMP-00010,Winter,Email,61263,1153,1623.56,False,EM,NaN
19,CMP-00020,Q1_Summer_CMP-00020,Summer,Email,6311,256,353.99,True,EM,NaN
20,CMP-00021,Q3_BlackFriday_CMP-00021,BlackFriday,Facebook,84104,2539,1975.49,True,FA,NaN
21,CMP-00022,Q2_BlackFriday_CMP-00022,BlackFriday,Google Ads,54707,1735,2428.40,True,GO,NaN
...,...,...,...,...,...,...,...,...,...,...
1911,CMP-01912,Q1_Launch_CMP-01912,Launch,Instagram,21581,573,931.41,True,IN,NaN
1953,CMP-01954,Q2_Summer_CMP-01954,Summer,Instagram,52047,2228,2214.29,False,IN,NaN
1971,CMP-01972,Q3_Winter_CMP-01972,Winter,Email,47163,2337,2185.99,True,EM,NaN
1981,CMP-01982,Q3_BlackFriday_CMP-01982,BlackFriday,Facebook,22288,552,926.14,False,FA,NaN


In [100]:
pd.crosstab(
    df_clean["channel"],
    df_clean["conversions"].isna(),
    margins=True
)

conversions,False,True,All
channel,,,
Email,341,34,375
Facebook,362,39,401
Google Ads,319,35,354
Instagram,314,44,358
Tiktok,372,40,412
Unknown,92,8,100
All,1800,200,2000


In [101]:
pd.crosstab(
    df_clean["active"],
    df_clean["conversions"].isna(),
    margins=True
)

conversions,False,True,All
active,,,
False,736,82,818
True,1064,118,1182
All,1800,200,2000


In [102]:
pd.crosstab(
    df_clean["campaign_type"],
    df_clean["conversions"].isna(),
    margins=True
)

conversions,False,True,All
campaign_type,,,
BlackFriday,451,52,503
Launch,451,55,506
Summer,435,44,479
Winter,463,49,512
All,1800,200,2000


In [103]:
conversion_missing[
    ["clicks_1", "impressions", "spend", "conversions"]
].describe()

,clicks_1,impressions,spend,conversions
count,200.000000,200.000000,200.000000,0.0
mean,1586.770000,51234.865000,4464.146700,NaN
std,1095.268255,28690.049332,35252.969147,NaN
min,26.000000,1077.000000,-2503.310000,NaN
25%,786.250000,26786.000000,908.795000,NaN
50%,1413.500000,48577.500000,1508.375000,NaN
75%,2207.750000,76668.250000,2722.710000,NaN
max,4618.000000,99875.000000,500000.000000,NaN


In [104]:
conversion_missing["clicks_1"].describe()

,clicks_1
count,200.000000
mean,1586.770000
std,1095.268255
min,26.000000
25%,786.250000
50%,1413.500000
75%,2207.750000
max,4618.000000


In [105]:
df_clean["conversion_missing_flag"] = (
    df_clean["conversions"].isna()
)

In [106]:
for col in ["channel", "campaign_tag", "active"]:
    summary = (
        df_clean.groupby(col, dropna=False)
        ["conversion_missing_flag"]
        .agg(
            total_records="size",
            missing_conversions="sum",
            missing_rate="mean"
        )
    )

    summary["missing_rate"] = (
        summary["missing_rate"] * 100
    ).round(2)

    print(f"\nMissing conversion analysis by {col}")
    display(summary)


Missing conversion analysis by channel


,total_records,missing_conversions,missing_rate
channel,,,
Email,375,34,9.07
Facebook,401,39,9.73
Google Ads,354,35,9.89
Instagram,358,44,12.29
Tiktok,412,40,9.71
Unknown,100,8,8.00



Missing conversion analysis by campaign_tag


,total_records,missing_conversions,missing_rate
campaign_tag,,,
EM,375,34,9.07
FA,401,39,9.73
GO,354,35,9.89
IN,358,44,12.29
TI,412,40,9.71
Unknown,100,8,8.00



Missing conversion analysis by active


,total_records,missing_conversions,missing_rate
active,,,
False,818,82,10.02
True,1182,118,9.98


**Conversion Missingness - Preliminary Finding**

The dataset contains 200 missing conversion values (9.90% of records). Initial analysis by marketing channel shows broadly similar missing rates across the main channels, with Instagram showing a somewhat higher rate. No cause can be established from this comparison alone.

Further investigation across campaign tags, active status, dates, clicks, impressions and spend is required before concluding whether missingness is associated with observable campaign characteristics. As the underlying source system is unavailable, the original cause of the missing values remains unknown.


In [107]:
df_clean.groupby("conversion_missing_flag")[
    ["impressions", "clicks_1", "spend"]
].agg(["count", "median", "mean", "std"]).round(2)

impressions                              clicks_1  \
                              count   median      mean       std    count   
conversion_missing_flag                                                     
False                          1800  50145.5  49696.16  28597.72     1800   
True                            200  48577.5  51234.86  28690.05      200   

                                                  spend                    \
                         median     mean      std count   median     mean   
conversion_missing_flag                                                     
False                    1231.5  1491.73  1085.70  1800  1411.02  2957.05   
True                     1413.5  1586.77  1095.27   200  1508.38  4464.15   

                                   
                              std  
conversion_missing_flag            
False                    23518.77  
True                     35252.97

In [108]:
# Preserving unknown conversion values
df_clean["conversion_missing_flag"] = (
    df_clean["conversions"].isna()
)

# Validating missingness before continuing
print("Rows:", len(df_clean))
print("Missing conversions:", df_clean["conversions"].isna().sum())
print(
    "Missing conversion percentage:",
    round(df_clean["conversions"].isna().mean() * 100, 2)
)

Rows: 2000
Missing conversions: 200
Missing conversion percentage: 10.0


I identified 200 missing values in the conversions column, representing approximately 9.9% of the dataset. Since the dataset did not provide sufficient information to determine whether these values represented zero conversions, tracking failures or unavailable records, I avoided making an unsupported assumption.

I preserved the original missing values and created a conversion status indicator to distinguish observed conversion counts from unknown values. This allowed me to retain all campaign records for analyses where the required data was available, while preventing unknown conversion counts from being misrepresented as actual outcomes.

I documented this limitation because the missing conversion data could affect conversion-related KPIs. In a production environment, I would investigate the source system and validate the cause with the data owner before deciding on a permanent treatment.


In [109]:
df_clean["campaign_tag"].unique()

<StringArray>
['TI', 'FA', 'EM', 'IN', 'GO', 'Unknown']
Length: 6, dtype: string

In [110]:
df_clean["start_date"].head(20)

,start_date
0,2023-11-24 00:00:00
1,2023-05-06 00:00:00
2,2023-12-13 00:00:00
3,2023-10-30
4,2023-04-22 00:00:00
5,2023-10-15 00:00:00
6,2023-10-07 00:00:00
7,2023-05-23
8,2023-03-23 00:00:00
9,2023-03-21 00:00:00


In [111]:
df_clean["start_date"].sample(20, random_state=42)

,start_date
1860,2023-08-07
353,2023-07-06
1333,2023-12-02 00:00:00
905,2023-03-14 00:00:00
1289,2023-04-25 00:00:00
1273,2023-05-21 00:00:00
938,2023-09-09 00:00:00
1731,2023-09-25
65,2023-10-04 00:00:00
1323,2023-08-22 00:00:00


In [112]:
df_clean["start_date"].isna().sum()

np.int64(0)

In [113]:
df_clean["start_date"].value_counts().reset_index()
# Findings from df["start_date"].value_counts().reset_index():
# There are 624 unique start dates.
# The most frequent start dates are '2023-10-15 00:00:00' and '2023-02-19 00:00:00', each appearing 12 times.
# The dates are in mixed formats: some are 'YYYY-MM-DD HH:MM:SS' and others are 'DD/MM/YYYY'.

,start_date,count
0,2023-02-19 00:00:00,12
1,2023-10-15 00:00:00,12
2,2023-08-15 00:00:00,11
3,2023-04-24 00:00:00,11
4,2023-10-20 00:00:00,10
...,...,...
619,2023-10-19,1
620,10/04/2023,1
621,10/11/2023,1
622,2023-06-02,1


In [114]:
# 1. copying the raw column
df_clean["start_date_raw"] = df_clean["start_date"].copy()

# 2. Converting to string and clean whitespace
s = df_clean["start_date"].astype(str).str.strip()

# 3. Removing unnecessary time component without regex
s = s.str.replace(" 00:00:00", "", regex=False)

# 4. As per known date formats, creating them in check the sum.
iso_mask = s.str.match(r"^\d{4}-\d{2}-\d{2}$")
dmy_mask = s.str.match(r"^\d{2}/\d{2}/\d{4}$")

# 5. Check what formats exist
print("YYYY-MM-DD:", iso_mask.sum())
print("DD/MM/YYYY:", dmy_mask.sum())
print("Unrecognized:", (~(iso_mask | dmy_mask)).sum())

# 6. Converting YYYY-MM-DD
df_clean.loc[iso_mask, "start_date"] = pd.to_datetime(
    s[iso_mask],
    format="%Y-%m-%d"
)

# 7. Converting DD/MM/YYYY
df_clean.loc[dmy_mask, "start_date"] = pd.to_datetime(
    s[dmy_mask],
    format="%d/%m/%Y"
)

# 8. Converting column to datetime
df_clean["start_date"] = pd.to_datetime(df_clean["start_date"])

# 9. Verifying dtype
print(df_clean["start_date"].dtype)

YYYY-MM-DD: 1816
DD/MM/YYYY: 184
Unrecognized: 0
datetime64[ns]


In [115]:
#Checking again for any inconsistencies in start date column

s = df_clean["start_date"].astype(str).str.strip()

print("With time:", s.str.contains("00:00:00", na=False).sum())
print("With slash:", s.str.contains("/", na=False).sum())
print("With hyphen:", s.str.contains("-", na=False).sum())

With time: 0
With slash: 0
With hyphen: 2000


In [116]:
# checking dtype for both start date and end date
df_clean[["start_date", "end_date"]].dtypes

,0
start_date,datetime64[ns]
end_date,object


In [117]:
#checking end_date columns for mixed dates

df_clean["start_date"].sample(20, random_state=42)

,start_date
1860,2023-08-07
353,2023-07-06
1333,2023-12-02
905,2023-03-14
1289,2023-04-25
1273,2023-05-21
938,2023-09-09
1731,2023-09-25
65,2023-10-04
1323,2023-08-22


In [118]:
#checking if missing date exist in end_date
df_clean["end_date"].isna().sum()

np.int64(0)

In [119]:
# Inspecting the end date column for inconsistencies
df_clean["end_date"].value_counts().reset_index()
# Findings from df["end_date"].value_counts().reset_index():
# There are 387 unique end dates.
# The most frequent end dates are '2023-08-27' (13 times), '2023-12-18' (12 times), '2023-07-06' (12 times),
# '2023-12-28' (12 times), and '2023-05-12' (12 times).
# All end dates are now in 'YYYY-MM-DD' format, indicating successful conversion to datetime objects.

,end_date,count
0,2023-05-12,12
1,2023-12-28,12
2,2023-12-18,12
3,2023-07-06,12
4,2023-03-19,11
...,...,...
382,2024-01-28,1
383,2024-01-21,1
384,2024-01-30,1
385,2023-01-10,1


In [120]:
# Preserving the current end-date values before reprocessing
df_clean["end_date_raw"] = df_clean["end_date"].copy()

end_dates = (
    df_clean["end_date_raw"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+00:00:00$", "", regex=True)
)

# Classifying the visible date formats
end_date_formats = pd.Series("Unrecognized", index=df_clean.index)

end_date_formats[end_dates.str.match(r"^\d{4}-\d{2}-\d{2}$", na=False)] = "YYYY-MM-DD"
end_date_formats[end_dates.str.match(r"^\d{4}/\d{2}/\d{2}$", na=False)] = "YYYY/MM/DD"
end_date_formats[end_dates.str.match(r"^\d{2}/\d{2}/\d{4}$", na=False)] = "DD/MM/YYYY or MM/DD/YYYY"
end_date_formats[end_dates.str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)] = "DD-MM-YYYY or MM-DD-YYYY"

print(end_date_formats.value_counts(dropna=False))

YYYY-MM-DD    2000
Name: count, dtype: int64


In [122]:
# Inspecting slash-separated dates where day/month interpretation may be ambiguous
ambiguous_end_dates = df_clean.loc[
    end_date_formats.eq("DD/MM/YYYY or MM/DD/YYYY"),
    ["campaign_id", "end_date_raw"]
]

display(ambiguous_end_dates.head(30))

# Inspecting anything that did not match our known patterns
display(
    df_clean.loc[
        end_date_formats.eq("Unrecognized"),
        ["campaign_id", "end_date_raw"]
    ].head(30)
)

,campaign_id,end_date_raw


,campaign_id,end_date_raw


In [121]:
# Validating the start date column.

formats = {
    "%Y-%m-%d": "YYYY-MM-DD",
    "%d/%m/%Y": "DD/MM/YYYY",
    "%m/%d/%Y": "MM/DD/YYYY",
    "%Y/%m/%d": "YYYY/MM/DD",
    "%d-%m-%Y": "DD-MM-YYYY",
    "%m-%d-%Y": "MM-DD-YYYY"
}

def detect_format(value):
    for fmt, label in formats.items():
        try:
            pd.to_datetime(value, format=fmt)
            return label
        except:
            pass

    return "Invalid/Unknown"


df_clean["start_date_format"] = df_clean["start_date"].apply(detect_format)

df_clean["start_date_format"].value_counts()

,count
start_date_format,
YYYY-MM-DD,2000


In [123]:
#Validating the end date column.

formats = {
    "%Y-%m-%d": "YYYY-MM-DD",
    "%d/%m/%Y": "DD/MM/YYYY",
    "%m/%d/%Y": "MM/DD/YYYY",
    "%Y/%m/%d": "YYYY/MM/DD",
    "%d-%m-%Y": "DD-MM-YYYY",
    "%m-%d-%Y": "MM-DD-YYYY"
}

def detect_format(value):
    for fmt, label in formats.items():
        try:
            pd.to_datetime(value, format=fmt)
            return label
        except:
            pass

    return "Invalid/Unknown"


df_clean["end_date_format"] = df_clean["end_date"].apply(detect_format)

df_clean["end_date_format"].value_counts()

,count
end_date_format,
YYYY-MM-DD,2000


In [124]:
#checking if end_date < start_date

wrong_end_date = df_clean["end_date"] < df_clean["start_date"]
print(df_clean.loc[wrong_end_date, ["campaign_id", "start_date", "end_date"]].head(5))

    campaign_id start_date    end_date
23    CMP-00024 2023-05-06  2023-05-01
54    CMP-00055 2023-09-01  2023-08-27
71    CMP-00072 2023-02-01  2023-01-27
124   CMP-00125 2023-04-01  2023-02-02
156   CMP-00157 2023-12-06  2023-12-01


In [126]:
#Converting start date and end date to datetime

df_clean["start_date"] = pd.to_datetime(df_clean["start_date"])
df_clean["end_date"] = pd.to_datetime(df_clean["end_date"])

In [127]:
#Checking for campaign duration in days

valid_dates = df_clean["end_date"] >= df_clean["start_date"]

campaign_duration = (
    df_clean.loc[valid_dates, "end_date"]
    - df_clean.loc[valid_dates, "start_date"]
).dt.days

campaign_duration.describe()

,0
count,1922.000000
mean,17.039022
std,19.859738
min,1.000000
25%,8.000000
50%,16.000000
75%,23.000000
max,285.000000


In [128]:
#checking the campaign median ( will check this against campaign_type median)
campaign_duration.median()

16.0

In [129]:
df_clean["campaign_duration_days"] = (
    df_clean["end_date"] - df_clean["start_date"]
).dt.days

In [130]:
df_clean["valid_duration_flag"] = (
    df_clean["campaign_duration_days"] >= 0
)

In [131]:
df_clean.loc[
    df_clean["valid_duration_flag"],
    "campaign_duration_days"
]

,campaign_duration_days
0,19
1,6
2,7
3,4
4,1
...,...
1995,17
1996,12
1997,14
1998,24


In [132]:
#Checking the spread by valid dates and campaign type.
df_clean.loc[valid_dates].groupby("campaign_type")[
    "campaign_duration_days"
].agg(["count", "median", "mean"])

,count,median,mean
campaign_type,,,
BlackFriday,482,16.0,16.663900
Launch,491,15.0,16.454175
Summer,462,15.0,16.073593
Winter,487,16.0,18.915811


In [133]:
#Checking the data where end_date < start_date
invalid_date_order = df_clean["end_date"] < df_clean["start_date"]
df_clean.loc[
    invalid_date_order,
    [
        "campaign_id",
        "campaign_name",
        "start_date",
        "end_date",
        "spend",
        "channel",
        "impressions",
        "clicks_1",
        "conversions"
    ]
]

,campaign_id,campaign_name,start_date,end_date,spend,channel,impressions,clicks_1,conversions
23,CMP-00024,Q4_Winter_CMP-00024,2023-05-06,2023-05-01,1793.94,Instagram,29693,995,179.0
54,CMP-00055,Q1_Launch_CMP-00055,2023-09-01,2023-08-27,563.31,Google Ads,18159,471,33.0
71,CMP-00072,Q1_Summer_CMP-00072,2023-02-01,2023-01-27,2090.74,Email,66697,1340,NaN
124,CMP-00125,Q4_Summer_CMP-00125,2023-04-01,2023-02-02,6449.30,Email,80575,3592,589.0
156,CMP-00157,Q3_Launch_CMP-00157,2023-12-06,2023-12-01,4690.79,Tiktok,82734,2912,NaN
...,...,...,...,...,...,...,...,...,...
1860,CMP-01861,Q4_Winter_CMP-01861,2023-08-07,2023-07-11,1232.12,Instagram,24019,1078,95.0
1901,CMP-01902,Q4_Summer_CMP-01902,2023-06-02,2023-02-15,460.85,Instagram,31015,457,61.0
1906,CMP-01907,Q3_Launch_CMP-01907,2023-07-04,2023-04-13,130.92,Facebook,7154,148,15.0
1941,CMP-01942,Q1_Winter_CMP-01942,2023-07-12,2023-07-07,2926.28,Google Ads,53326,2057,379.0


In [134]:
#Storing the invalid dates as a flag
df_clean["invalid_date_order_flag"] = (
    df_clean["end_date"] < df_clean["start_date"]
)

In [135]:
#isolating the invalid dates to investigate further
invalid_date = df_clean[df_clean["invalid_date_order_flag"]].copy()

In [136]:
# About 3.9% of campaigns have End_Date < Start_Date.
df_clean["invalid_date_order_flag"].value_counts()

,count
invalid_date_order_flag,
False,1922
True,78


In [137]:
# Calculating how many days the end date is before the start date

invalid_date_records = df_clean.loc[
    invalid_date_order,
    [
        "campaign_id",
        "campaign_name",
        "start_date",
        "end_date",
        "channel",
        "spend"
    ]
].copy()

invalid_date_records["date_difference_days"] = (
    invalid_date_records["end_date"]
    - invalid_date_records["start_date"]
).dt.days

invalid_date_records["date_difference_days"].value_counts().sort_index()

,count
date_difference_days,
-294,1
-292,2
-289,1
-274,1
-264,1
-263,1
-238,1
-208,1
-201,1


In [138]:
invalid_date_records.sort_values(
    "date_difference_days"
)[[
    "campaign_id",
    "campaign_name",
    "start_date",
    "end_date",
    "date_difference_days",
    "channel",
    "spend"
]].head(20)

,campaign_id,campaign_name,start_date,end_date,date_difference_days,channel,spend
1504,CMP-01505,Q2_Winter_CMP-01505,2023-12-01,2023-02-10,-294,Tiktok,188.38
1775,CMP-01776,Q3_Winter_CMP-01776,2023-11-01,2023-01-13,-292,Google Ads,650.04
1814,CMP-01815,Q2_Winter_CMP-01815,2023-12-02,2023-02-13,-292,Tiktok,3097.66
893,CMP-00894,Q4_Winter_CMP-00894,2023-11-01,2023-01-16,-289,Tiktok,760.11
498,CMP-00499,Q2_BlackFriday_CMP-00499,2023-11-01,2023-01-31,-274,Google Ads,3793.11
915,CMP-00916,Q1_Launch_CMP-00916,2023-12-03,2023-03-14,-264,Instagram,714.88
241,CMP-00242,Q4_Summer_CMP-00242,2023-10-01,2023-01-11,-263,Instagram,2032.32
733,CMP-00734,Q1_BlackFriday_CMP-00734,2023-11-02,2023-03-09,-238,Google Ads,484.70
344,CMP-00345,Q1_BlackFriday_CMP-00345,2023-12-04,2023-05-10,-208,Tiktok,2629.55
1561,CMP-01562,Q2_Launch_CMP-01562,2023-11-04,2023-04-17,-201,Tiktok,1542.32


37 records show -5 days descripency and some dates show also show 298 days which needs to investigated further.

In [139]:
# Checking the spread of days between start and end date
invalid_date_records["date_difference_days"].describe()

,date_difference_days
count,78.000000
mean,-70.423077
std,88.446993
min,-294.000000
25%,-107.000000
50%,-21.500000
75%,-5.000000
max,-5.000000


In [140]:
#checking the count of reaccuring date difference between end date and start date where end date < start date
invalid_date_records["days_invalid_by"] = (
    invalid_date_records["start_date"]
    - invalid_date_records["end_date"]
).dt.days

invalid_date_records["days_invalid_by"].value_counts().sort_index()

,count
days_invalid_by,
5,37
13,1
16,1
27,2
32,1
35,1
38,1
58,1
63,1


In [141]:
# Checking the records for 5 day difference to check any relation with campaigns.
five_day_errors = invalid_date_records[
    invalid_date_records["date_difference_days"] == -5
]

five_day_errors[
    ["campaign_id", "campaign_name", "start_date", "end_date"]
]

,campaign_id,campaign_name,start_date,end_date
23,CMP-00024,Q4_Winter_CMP-00024,2023-05-06,2023-05-01
54,CMP-00055,Q1_Launch_CMP-00055,2023-09-01,2023-08-27
71,CMP-00072,Q1_Summer_CMP-00072,2023-02-01,2023-01-27
156,CMP-00157,Q3_Launch_CMP-00157,2023-12-06,2023-12-01
200,CMP-00201,Q3_BlackFriday_CMP-00201,2023-01-11,2023-01-06
218,CMP-00219,Q3_Winter_CMP-00219,2023-10-03,2023-09-28
221,CMP-00222,Q1_Winter_CMP-00222,2023-08-26,2023-08-21
280,CMP-00281,Q4_Summer_CMP-00281,2023-12-10,2023-12-05
303,CMP-00304,Q4_Winter_CMP-00304,2023-05-17,2023-05-12
346,CMP-00347,Q1_Summer_CMP-00347,2023-11-07,2023-11-02


In [142]:
# Checking the data for records of other than 5 days difference to check if there is any relation.
other_date_errors = invalid_date_records[
    invalid_date_records["date_difference_days"] != -5
]

other_date_errors[
    ["campaign_id", "campaign_name", "start_date", "end_date",
     "date_difference_days"]
]

,campaign_id,campaign_name,start_date,end_date,date_difference_days
124,CMP-00125,Q4_Summer_CMP-00125,2023-04-01,2023-02-02,-58
195,CMP-00196,Q4_Summer_CMP-00196,2023-08-04,2023-04-21,-105
241,CMP-00242,Q4_Summer_CMP-00242,2023-10-01,2023-01-11,-263
258,CMP-00259,Q2_Summer_CMP-00259,2023-05-03,2023-03-26,-38
323,CMP-00324,Q1_Winter_CMP-00324,2023-09-07,2023-08-06,-32
344,CMP-00345,Q1_BlackFriday_CMP-00345,2023-12-04,2023-05-10,-208
353,CMP-00354,Q3_BlackFriday_CMP-00354,2023-07-06,2023-06-20,-16
380,CMP-00381,Q2_BlackFriday_CMP-00381,2023-05-01,2023-01-17,-104
395,CMP-00396,Q3_Summer_CMP-00396,2023-05-02,2023-02-14,-77
454,CMP-00455,Q3_Winter_CMP-00455,2023-12-09,2023-09-20,-80


In [143]:
#Checking the number of large date anomalies
large_date_anomalies = invalid_date_records[
    invalid_date_records["date_difference_days"] <= -200
].copy()

large_date_anomalies.sort_values(
    "date_difference_days"
)[[
    "campaign_id",
    "campaign_name",
    "start_date",
    "end_date",
    "date_difference_days",
    "channel",
    "spend"
]]

,campaign_id,campaign_name,start_date,end_date,date_difference_days,channel,spend
1504,CMP-01505,Q2_Winter_CMP-01505,2023-12-01,2023-02-10,-294,Tiktok,188.38
1775,CMP-01776,Q3_Winter_CMP-01776,2023-11-01,2023-01-13,-292,Google Ads,650.04
1814,CMP-01815,Q2_Winter_CMP-01815,2023-12-02,2023-02-13,-292,Tiktok,3097.66
893,CMP-00894,Q4_Winter_CMP-00894,2023-11-01,2023-01-16,-289,Tiktok,760.11
498,CMP-00499,Q2_BlackFriday_CMP-00499,2023-11-01,2023-01-31,-274,Google Ads,3793.11
915,CMP-00916,Q1_Launch_CMP-00916,2023-12-03,2023-03-14,-264,Instagram,714.88
241,CMP-00242,Q4_Summer_CMP-00242,2023-10-01,2023-01-11,-263,Instagram,2032.32
733,CMP-00734,Q1_BlackFriday_CMP-00734,2023-11-02,2023-03-09,-238,Google Ads,484.70
344,CMP-00345,Q1_BlackFriday_CMP-00345,2023-12-04,2023-05-10,-208,Tiktok,2629.55
1561,CMP-01562,Q2_Launch_CMP-01562,2023-11-04,2023-04-17,-201,Tiktok,1542.32


In [144]:
#Checking the highest date anomaly

large_date_anomalies.iloc[0]

worst_campaign = large_date_anomalies.sort_values(
    "date_difference_days"
).iloc[0]

worst_campaign

,1504
campaign_id,CMP-01505
campaign_name,Q2_Winter_CMP-01505
start_date,2023-12-01 00:00:00
end_date,2023-02-10 00:00:00
channel,Tiktok
spend,188.38
date_difference_days,-294
days_invalid_by,294


In [145]:
#Checking if any other campaigns were run during the same period as the highest date anomaly

hypothetical_start = worst_campaign["end_date"]
hypothetical_end = worst_campaign["start_date"]

print("Hypothetical Period:")
print("Start:", hypothetical_start)
print("End:", hypothetical_end)

Hypothetical Period:
Start: 2023-02-10 00:00:00
End: 2023-12-01 00:00:00


In [146]:
#Checking overlapping period to the hypothetical period created to fix the highest date anomaly
overlapping_campaigns = df_clean[
    (df_clean["start_date"] <= hypothetical_end) &
    (df_clean["end_date"] >= hypothetical_start) &
    (df_clean["campaign_id"] != worst_campaign["campaign_id"])
].copy()

In [147]:
overlapping_campaigns[
    [
        "campaign_id",
        "campaign_name",
        "start_date",
        "end_date",
        "channel",
        "spend"
    ]
].sort_values("start_date").head(20)

,campaign_id,campaign_name,start_date,end_date,channel,spend
1389,CMP-01390,Q2_Winter_CMP-01390,2023-01-07,2023-07-06,Email,4208.98
294,CMP-00295,Q2_Summer_CMP-00295,2023-01-08,2023-08-13,Google Ads,227.86
551,CMP-00552,Q2_Launch_CMP-00552,2023-01-09,2023-09-17,Instagram,68.36
1016,CMP-01017,Q2_Summer_CMP-01017,2023-01-09,2023-09-05,Email,1837.35
763,CMP-00764,Q2_Winter_CMP-00764,2023-01-09,2023-09-24,Email,488.36
196,CMP-00197,Q3_Launch_CMP-00197,2023-01-10,2023-10-15,Instagram,2992.62
1295,CMP-01296,Q3_Winter_CMP-01296,2023-01-11,2023-02-10,Facebook,632.04
1229,CMP-01230,Q2_Summer_CMP-01230,2023-01-15,2023-02-14,Facebook,6356.83
199,CMP-00200,Q2_BlackFriday_CMP-00200,2023-01-16,2023-02-13,Google Ads,3422.33
1761,CMP-01762,Q4_Winter_CMP-01762,2023-01-18,2023-02-16,Google Ads,1915.77


In [148]:
#Excluding the other invalid dates from the check to make sure the analysis doesn't distort

valid_campaigns = df_clean[
    df_clean["end_date"] >= df_clean["start_date"]
].copy()

In [149]:
overlapping_campaigns = valid_campaigns[
    (valid_campaigns["start_date"] <= hypothetical_end) &
    (valid_campaigns["end_date"] >= hypothetical_start) &
    (valid_campaigns["campaign_id"] != worst_campaign["campaign_id"])
].copy()

overlapping_campaigns[
    [
        "campaign_id",
        "campaign_name",
        "start_date",
        "end_date",
        "channel",
        "spend"
    ]
].sort_values("start_date").head(20)

,campaign_id,campaign_name,start_date,end_date,channel,spend
1389,CMP-01390,Q2_Winter_CMP-01390,2023-01-07,2023-07-06,Email,4208.98
294,CMP-00295,Q2_Summer_CMP-00295,2023-01-08,2023-08-13,Google Ads,227.86
763,CMP-00764,Q2_Winter_CMP-00764,2023-01-09,2023-09-24,Email,488.36
1016,CMP-01017,Q2_Summer_CMP-01017,2023-01-09,2023-09-05,Email,1837.35
551,CMP-00552,Q2_Launch_CMP-00552,2023-01-09,2023-09-17,Instagram,68.36
196,CMP-00197,Q3_Launch_CMP-00197,2023-01-10,2023-10-15,Instagram,2992.62
1295,CMP-01296,Q3_Winter_CMP-01296,2023-01-11,2023-02-10,Facebook,632.04
1229,CMP-01230,Q2_Summer_CMP-01230,2023-01-15,2023-02-14,Facebook,6356.83
199,CMP-00200,Q2_BlackFriday_CMP-00200,2023-01-16,2023-02-13,Google Ads,3422.33
1761,CMP-01762,Q4_Winter_CMP-01762,2023-01-18,2023-02-16,Google Ads,1915.77


In [150]:
#Checking campaign duration in days by campaign type.
df_clean.loc[valid_dates].groupby("campaign_type")["campaign_duration_days"].agg(
    ["count", "median", "mean"]
)

,count,median,mean
campaign_type,,,
BlackFriday,482,16.0,16.663900
Launch,491,15.0,16.454175
Summer,462,15.0,16.073593
Winter,487,16.0,18.915811


### Datetime Standardization Note

During datetime standardization, the `Start_Date` field was found to contain inconsistent date representations, including `DD/MM/YYYY` and `YYYY/MM/DD` formats, with some records also containing time components. Rather than coercing unparseable values to missing, the raw values were retained and a format-aware parsing approach was applied - `YYYY/MM/DD`

Ambiguous dates, such as `01/02/2023`, were separately identified for validation rather than arbitrarily assigning a date format. Since this is a Kaggle dataset and the underlying source systems are not directly accessible, the original source records could not be consulted to definitively resolve all ambiguous values. If source-level information were available, the relevant source systems or data documentation would be reviewed to establish the intended date convention before making any assumptions in terms of timezones and system generated errors.


Date-order anomalies: 78 records had an end date earlier than the start date. Rather than automatically imputing or deleting these records, I investigated the magnitude and contextual patterns. Ten records had gaps of 200+ days. Campaign overlap was observed in the dataset, so overlap alone was insufficient to determine the correct dates. The anomalous records were therefore retained and flagged pending a reliable business rule or source for date correction.

In [154]:
df_clean.head(10)

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks_1,spend,conversions,active,...,channel_missing_flag,conversion_missing_flag,start_date_raw,end_date_raw,start_date_format,end_date_format,campaign_duration_days,valid_duration_flag,invalid_date_order_flag,spend_clean
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,Tiktok,16795,197,102.82,20.0,True,...,False,False,2023-11-24 00:00:00,2023-12-13,YYYY-MM-DD,YYYY-MM-DD,19,True,False,102.82
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,False,...,False,False,2023-05-06 00:00:00,2023-05-12,YYYY-MM-DD,YYYY-MM-DD,6,True,False,24.33
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,False,...,False,False,2023-12-13 00:00:00,2023-12-20,YYYY-MM-DD,YYYY-MM-DD,7,True,False,1323.39
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,Tiktok,55886,2019,2180.38,135.0,True,...,False,False,2023-10-30,2023-11-03,YYYY-MM-DD,YYYY-MM-DD,4,True,False,2180.38
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,...,False,False,2023-04-22 00:00:00,2023-04-23,YYYY-MM-DD,YYYY-MM-DD,1,True,False,252.44
5,CMP-00006,Q4_BlackFriday_CMP-00006,2023-10-15,2023-10-28,Instagram,83386,2643,2697.03,NaN,True,...,False,True,2023-10-15 00:00:00,2023-10-28,YYYY-MM-DD,YYYY-MM-DD,13,True,False,2697.03
6,CMP-00007,Q3_Launch_CMP-00007,2023-10-07,2023-10-23,Facebook,38194,1135,1232.76,178.0,True,...,False,False,2023-10-07 00:00:00,2023-10-23,YYYY-MM-DD,YYYY-MM-DD,16,True,False,1232.76
7,CMP-00008,Q4_Launch_CMP-00008,2023-05-23,2023-05-28,Instagram,88498,1173,865.70,127.0,True,...,False,False,2023-05-23,2023-05-28,YYYY-MM-DD,YYYY-MM-DD,5,True,False,865.70
8,CMP-00009,Q4_BlackFriday_CMP-00009,2023-03-23,2023-04-01,Google Ads,45131,1179,1046.18,104.0,True,...,False,False,2023-03-23 00:00:00,2023-04-01,YYYY-MM-DD,YYYY-MM-DD,9,True,False,1046.18
9,CMP-00010,Q2_Winter_CMP-00010,2023-03-21,2023-04-01,Email,61263,1153,1623.56,NaN,False,...,False,True,2023-03-21 00:00:00,2023-04-01,YYYY-MM-DD,YYYY-MM-DD,11,True,False,1623.56


In [157]:
print("Clicks > Impressions:")
print((df_clean["clicks_1"] > df_clean["impressions"]).sum())

print("\nConversions > Clicks:")
print((df_clean["conversions"] > df_clean["clicks_1"]).sum())

print("\nNegative Impressions:")
print((df_clean["impressions"] < 0).sum())

print("\nNegative Clicks:")
print((df_clean["clicks_1"] < 0).sum())

print("\nNegative Conversions:")
print((df_clean["conversions"] < 0).sum())

Clicks > Impressions:
0

Conversions > Clicks:
0

Negative Impressions:
0

Negative Clicks:
0

Negative Conversions:
0


In [158]:
#checking for wrong clicks
df_clean["clicks_gt_impressions_flag"] = (
    df_clean["clicks_1"] > df_clean["impressions"]
)

In [159]:
# Checking clicks flag
df_clean["conversions_gt_clicks_flag"] = (
    df_clean["conversions"] > df_clean["clicks_1"]
)

In [160]:
# Checking for negative metrics
df_clean["negative_impressions_flag"] = (
    df_clean["impressions"] < 0
)

df_clean["negative_clicks_flag"] = (
    df_clean["clicks_1"] < 0
)

df_clean["negative_conversions_flag"] = (
    df_clean["conversions"] < 0
)

In [161]:
quality_flags = {
    "Clicks > Impressions": df_clean["clicks_gt_impressions_flag"].sum(),
    "Conversions > Clicks": df_clean["conversions_gt_clicks_flag"].sum(),
    "Negative Impressions": df_clean["negative_impressions_flag"].sum(),
    "Negative Clicks": df_clean["negative_clicks_flag"].sum(),
    "Negative Conversions": df_clean["negative_conversions_flag"].sum()
}

pd.Series(quality_flags)

,0
Clicks > Impressions,0
Conversions > Clicks,0
Negative Impressions,0
Negative Clicks,0
Negative Conversions,0


In [162]:
# checking the negative metric funnel

df_clean.loc[
    df_clean["clicks_gt_impressions_flag"],
    [
        "campaign_id",
        "campaign_name",
        "impressions",
        "clicks_1",
        "conversions",
        "spend",
        "channel"
    ]
]

,campaign_id,campaign_name,impressions,clicks_1,conversions,spend,channel


In [163]:
df_clean.loc[
    df_clean["conversions_gt_clicks_flag"],
    [
        "campaign_id",
        "campaign_name",
        "impressions",
        "clicks_1",
        "conversions",
        "spend",
        "channel"
    ]
]

,campaign_id,campaign_name,impressions,clicks_1,conversions,spend,channel


In [164]:
#Creating a conversion missingness flag

df_clean["conversion_missing_flag"] = (
    df_clean["conversions"].isna()
)

In [165]:
df_clean["conversion_missing_flag"].value_counts()

,count
conversion_missing_flag,
False,1800
True,200


In [166]:
df_clean.loc[
    df_clean["conversion_missing_flag"],
    [
        "campaign_id",
        "campaign_name",
        "channel",
        "impressions",
        "clicks_1",
        "spend",
        "active",
        "campaign_tag"
    ]
].head(20)

,campaign_id,campaign_name,channel,impressions,clicks_1,spend,active,campaign_tag
5,CMP-00006,Q4_BlackFriday_CMP-00006,Instagram,83386,2643,2697.03,True,IN
9,CMP-00010,Q2_Winter_CMP-00010,Email,61263,1153,1623.56,False,EM
19,CMP-00020,Q1_Summer_CMP-00020,Email,6311,256,353.99,True,EM
20,CMP-00021,Q3_BlackFriday_CMP-00021,Facebook,84104,2539,1975.49,True,FA
21,CMP-00022,Q2_BlackFriday_CMP-00022,Google Ads,54707,1735,2428.40,True,GO
48,CMP-00049,Q4_Winter_CMP-00049,Facebook,44001,1483,2928.86,False,FA
52,CMP-00053,Q1_Winter_CMP-00053,Facebook,24483,883,640.98,False,FA
66,CMP-00067,Q1_BlackFriday_CMP-00067,Google Ads,41757,1535,1772.36,True,GO
71,CMP-00072,Q1_Summer_CMP-00072,Email,66697,1340,2090.74,True,EM
72,CMP-00073,Q3_Summer_CMP-00073,Google Ads,38065,927,1808.48,False,GO


In [167]:
df_clean.groupby("channel")["conversion_missing_flag"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
channel,,,
Email,375,34,0.090667
Facebook,401,39,0.097257
Google Ads,354,35,0.098870
Instagram,358,44,0.122905
Tiktok,412,40,0.097087
Unknown,100,8,0.080000


In [170]:
# Check the fields used in your existing KPI calculations
kpi_fields = [
    "impressions",
    "clicks_1",
    "conversions",
    "spend",
    "spend_analysis",
    "spend_clean",
    "ctr",
    "conversion_rate",
    "cpc",
    "cost_per_conversion"
]

print("Available KPI fields:")
print([col for col in kpi_fields if col in df_clean.columns])

# Checking how many campaigns have the required data for each metric
kpi_coverage = pd.DataFrame({
    "metric": [
        "CTR",
        "Conversion rate",
        "CPC",
        "Cost per conversion"
    ],
    "eligible_campaigns": [
        (df_clean["impressions"] > 0).sum(),
        (
            (df_clean["clicks_1"] > 0)
            & df_clean["conversions"].notna()
        ).sum(),
        (
            (df_clean["clicks_1"] > 0)
            & df_clean["spend_analysis"].notna()
        ).sum(),
        (
            (df_clean["conversions"] > 0)
            & df_clean["spend_analysis"].notna()
        ).sum()
    ]
})

kpi_coverage["eligible_percentage"] = (
    kpi_coverage["eligible_campaigns"] / len(df_clean) * 100
).round(2)

display(kpi_coverage)

Available KPI fields:
['impressions', 'clicks_1', 'conversions', 'spend', 'spend_analysis', 'spend_clean', 'ctr']


,metric,eligible_campaigns,eligible_percentage
0,CTR,2000,100.00
1,Conversion rate,1800,90.00
2,CPC,1981,99.05
3,Cost per conversion,1781,89.05


#Marketing Metrics

In [176]:
# CTR
df_clean["ctr"] = np.where(
    df_clean["impressions"] > 0,
    df_clean["clicks_1"] / df_clean["impressions"],
    np.nan
)

In [177]:
# Conversions Rate
df_clean["conversion_rate"] = np.where(
    df_clean["clicks_1"] > 0,
    df_clean["conversions"] / df_clean["clicks_1"],
    np.nan
)

In [173]:
# Cost per click
df_clean["cpc"] = np.where(
    df_clean["clicks_1"] > 0,
    df_clean["spend_analysis"] / df_clean["clicks_1"],
    np.nan
)

In [174]:
# Cost per Conversion
df_clean["cost_per_conversion"] = np.where(
    df_clean["conversions"] > 0,
    df_clean["spend_analysis"] / df_clean["conversions"],
    np.nan
)

In [175]:
df_clean["funnel_valid_flag"] = (
    (df_clean["impressions"] >= 0) &
    (df_clean["clicks_1"] >= 0) &
    (df_clean["conversions"] >= 0) &
    (df_clean["clicks_1"] <= df_clean["impressions"]) &
    (
        df_clean["conversions"].isna() |
        (df_clean["conversions"] <= df_clean["clicks_1"])
    )
)

In [180]:
#Which marketing channels generate the most engagement and conversions,
# and which channels use the budget most efficiently?

# Creating a channel-level performance summary
# Using the final cleaned spend field consistently.

channel_performance = (
    df_clean.groupby("channel", dropna=False)
    .agg(
        campaigns=("campaign_id", "nunique"),
        impressions=("impressions", "sum"),
        clicks=("clicks_1", "sum"),
        conversions=("conversions", "sum"),
        spend=("spend_clean", "sum"),
        campaigns_with_conversions=("conversions", "count")
    )
    .reset_index()
)

# Calculating aggregate KPIs from totals, not averages of campaign rates
channel_performance["ctr_pct"] = (
    channel_performance["clicks"]
    / channel_performance["impressions"].replace(0, float("nan"))
    * 100
).round(2)

channel_performance["conversion_rate_pct"] = (
    channel_performance["conversions"]
    / channel_performance["clicks"].replace(0, float("nan"))
    * 100
).round(2)

channel_performance["cpc"] = (
    channel_performance["spend"]
    / channel_performance["clicks"].replace(0, float("nan"))
).round(2)

channel_performance["cost_per_conversion"] = (
    channel_performance["spend"]
    / channel_performance["conversions"].replace(0, float("nan"))
).round(2)

display(
    channel_performance.sort_values(
        "spend", ascending=False
    )
)

,channel,campaigns,impressions,clicks,conversions,spend,campaigns_with_conversions,ctr_pct,conversion_rate_pct,cpc,cost_per_conversion
0,Email,375,19156770,560082,62632.0,2197742.66,341,2.92,11.18,3.92,35.09
1,Facebook,401,20395140,623589,69502.0,1776485.52,362,3.06,11.15,2.85,25.56
4,Tiktok,412,20528411,617274,67713.0,770686.97,372,3.01,10.97,1.25,11.38
2,Google Ads,354,17646113,533706,60384.0,682094.12,319,3.02,11.31,1.28,11.30
3,Instagram,358,17331450,517158,57088.0,635386.13,314,2.98,11.04,1.23,11.13
5,Unknown,100,4642169,150666,17744.0,172413.96,92,3.25,11.78,1.14,9.72
